# Quantum Hashing Algorithm

This is my version of hashing algorithm using unbiased coin operator for quantum walk. How this algorithm works is:
- Initialize unbiased coin operator with Hadamard gate
- Split binary string into chunks of 4 bits to be processed per hashing cycle
- Use coin operator to determine if we increment or decrement value of 4 bits i.e. taking a step back or forward
- Repeat the process X amount of steps, each time appending hex value of new 4 bit value
- Retrieve final hash

## Step 1 : Setup
This section is meant for setting up variables, initalizing our quantum circuit and any additional parameters required for hashing

In [ ]:
!pip install qiskit qiskit_aer matplotlib pylatexenc

In [4]:
from qiskit import QuantumCircuit
from qiskit_aer.primitives import SamplerV2
from qiskit_aer.primitives import Sampler
import warnings

#message = "test"
#message_bits = ''.join(format(ord(char), '08b') for char in message)

message_bits = "0100111001100100"

qbit_size = 8
hash_bits = list()

qcoin = QuantumCircuit(1)
qcoin.h(0)
qcoin.measure_all()

sampler = SamplerV2()

warnings.filterwarnings("ignore")

def shift_right(qbit_size) -> QuantumCircuit:
    inc = QuantumCircuit(qbit_size, name="inc")
    for target in range(qbit_size - 1, 0, -1):
        control = list(range(target))
        inc.mcx(control, target)
    inc.x(0)
    return inc

def shift_left(qbit_size) -> QuantumCircuit:
    dec = QuantumCircuit(qbit_size)
    for target in range(0, qbit_size, +1):
        control = list(range(target))
        dec.mcx(control, target)
    return dec

def qbit_measure(qc) -> dict:
    sampler = Sampler()
    job = sampler.run([qc], shots=1000)
    results = job.result()

    return results.quasi_dists[0]

# Step 2: Hash Algorithm

Setting up algorithm for creating the hash

In [22]:
def qbit_hash(message_bits) -> str:
    final_hash = ""

    for l in range(0, int(len(message_bits)/4)):
        for i in range(0, len(message_bits), 8):
            quant_walk = QuantumCircuit(qbit_size)
            position = 0
            
            for j in range(i, i+8):
                if message_bits[j] == "1":
                    quant_walk.x(position)
                position += 1             
        
            for k in range(256):      
                job = sampler.run([qcoin], shots=100)
                results = job.result()
                counts = results[0].data.meas.get_counts()
                
                if counts['1'] > counts['0']:
                    quant_walk.compose(shift_right(qbit_size), inplace=True)
                else:
                    quant_walk.compose(shift_left(qbit_size), inplace=True)  
                
            quant_walk.measure_all()
            measurement = qbit_measure(quant_walk)
            for key in measurement:
                    final_hash += hex(key)[2:]

    return final_hash

## Step 3: Results

Run hashing and get results

In [25]:
print(qbit_hash(message_bits))

6486c1c6c286e10
